In [2]:
# **确保 Notebook 能找到 `utils/` 目录**
module_path = os.path.abspath(os.path.join(os.getcwd(), "utils"))
if module_path not in sys.path:
    sys.path.append(module_path)

# **导入 predict_emotion**
from model_inference import predict_emotion


c:\Users\ziyan\anaconda3\envs\d2l\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] 找不到指定的程序。'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ 微调的 BERT + Wav2Vec2 模型加载成功！


In [1]:
import os
import sys
import io

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env 文件
SPEECH_KEY = os.getenv("AZURE_SPEECH_KEY")
SPEECH_REGION = os.getenv("AZURE_SPEECH_REGION")

print(f"SPEECH_KEY: {SPEECH_KEY}")  # 确保 API Key 存在
print(f"SPEECH_REGION: {SPEECH_REGION}")  # 确保 Region 正确


In [ ]:
import azure.cognitiveservices.speech as speechsdk

In [ ]:
# 需要合成的语音角色（所有支持这些情绪的 en-US 语音）
voice_names = voice_names = [
    "en-US-AriaNeural", "en-US-DavisNeural", "en-US-GuyNeural",
    "en-US-JaneNeural", "en-US-JasonNeural", "en-US-JennyNeural",
    "en-US-NancyNeural", "en-US-SaraNeural", "en-US-TonyNeural"
]

# **多个中性句子，测试 Joy vs. Sad**
test_sentences = { 
    "sentence16": "I’m not going to argue with you about this.",  
}

# **每个句子测试 快乐 vs. 悲伤**
emotions = {
    "sentence16":  ["angry"],  
}

In [ ]:

# 初始化 Azure 语音服务
speech_config = speechsdk.SpeechConfig(subscription=SPEECH_KEY, region=SPEECH_REGION)

def synthesize_speech(voice_name, text, emotion, sentence_id):
    # 生成正确的文件夹名称
    folder_name = f"{emotion}_{sentence_id}"  # 每个情绪单独一个文件夹
    os.makedirs(folder_name, exist_ok=True)

    # 生成 SSML 代码，放大情绪强度
    ssml_text = f"""
    <speak xmlns="http://www.w3.org/2001/10/synthesis"
           xmlns:mstts="http://www.w3.org/2001/mstts"
           xmlns:emo="http://www.w3.org/2009/10/emotionml"
           version="1.0" xml:lang="en-US">
        <voice name="{voice_name}">
            <s />
            <mstts:express-as style="{emotion}" styledegree="2.0">
                {text}
            </mstts:express-as>
        </voice>
    </speak>
    """

    # 语音文件路径（修正命名规则）
    file_name = f"{voice_name}_{emotion}_2x.wav"
    file_path = os.path.join(folder_name, file_name)

    # 配置语音输出
    audio_config = speechsdk.audio.AudioOutputConfig(filename=file_path)
    synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)

    # 进行语音合成
    result = synthesizer.speak_ssml_async(ssml_text).get()

    # 检查是否成功
    if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        print(f"✅ Saved: {file_path}")
    elif result.reason == speechsdk.ResultReason.Canceled:
        cancellation_details = speechsdk.SpeechSynthesisCancellationDetails(result)
        print(f"❌ Failed for {voice_name}: {cancellation_details.reason}")
        if cancellation_details.reason == speechsdk.CancellationReason.Error:
            print(f"🔴 Error Details: {cancellation_details.error_details}")

# **批量处理所有情绪语音**
for sentence_id, text in test_sentences.items():
    for emotion in emotions[sentence_id]:  # 逐个处理不同的情绪
        for voice in voice_names:
            synthesize_speech(voice, text, emotion, sentence_id)

print("🎉 All voices synthesized and saved successfully!")


In [18]:

# **需要测试的文件夹**
test_folders = [
    "cheerful_sentence9", "sad_sentence9",
    "cheerful_sentence10", "sad_sentence10",
    "cheerful_sentence11", "sad_sentence11",
    "cheerful_sentence12", "sad_sentence12",
    "cheerful_sentence13", "sad_sentence13"
]

# **对应的测试文本**
# **5 个容易触发机器 "Fear" 反应的句子**
test_sentences = {
    "sentence9":  "You have got to be kidding me.",  # Anger / Frustration
    "sentence10": "How many times do I have to say this?",  # Anger / Annoyance
    "sentence11": "This is completely unacceptable.",  # Anger / Disgust
    "sentence12": "I’m not going to argue with you about this.",  # Anger / Frustration
    "sentence13": "You really don’t care, do you?",  # Anger / Disappointment
}




In [19]:
def test_audio_emotions():
    print(f"🔍 开始测试 {len(test_folders)} 个文件夹中的音频...\n")

    # **遍历每个文件夹**
    for folder in test_folders:
        if not os.path.exists(folder):
            print(f"⚠️ 文件夹 `{folder}` 不存在，跳过...")
            continue

        # **获取句子 ID**
        sentence_id = folder.split("_")[-1]  # 提取 "sentence9" 这样的编号
        fixed_text = test_sentences.get(sentence_id, "Unknown sentence")  # 获取对应句子

        audio_files = [f for f in os.listdir(folder) if f.endswith(".wav")]
        print(f"\n📂 进入文件夹: {folder}，找到 {len(audio_files)} 个音频文件")
        print(f"📝 对应文本: \"{fixed_text}\"")

        # **测试每个音频**
        for audio_file in audio_files:
            audio_path = os.path.join(folder, audio_file)

            print(f"\n🎤 测试音频: {audio_file}")

            try:
                # **屏蔽 `predict_emotion()` 内部的 `print()`**
                original_stdout = sys.stdout
                sys.stdout = io.StringIO()

                emotions = predict_emotion(fixed_text, audio_path)  # 运行情绪预测

                sys.stdout = original_stdout  # 恢复标准输出

                # **解析情绪预测结果**
                if isinstance(emotions, dict):
                    sorted_emotions = sorted(emotions.items(), key=lambda x: float(x[1][:-1]), reverse=True)[:3]
                    top_emotions = ", ".join([f"{label}: {prob}" for label, prob in sorted_emotions])
                    print(f"📊 预测结果 (Top 3): {top_emotions}")
                else:
                    print(f"⚠️ 预测返回格式错误: {emotions}")

            except Exception as e:
                print(f"❌ 处理 {audio_file} 时出错: {e}")

    print("\n✅ 所有音频测试完成！")

# **执行测试**
test_audio_emotions()


🔍 开始测试 10 个文件夹中的音频...


📂 进入文件夹: cheerful_sentence9，找到 4 个音频文件
📝 对应文本: "You have got to be kidding me."

🎤 测试音频: en-US-AriaNeural_cheerful_2x.wav
📊 预测结果 (Top 3): Surprised: 79.28%, Fearful: 8.52%, Angry: 5.84%

🎤 测试音频: en-US-DavisNeural_cheerful_2x.wav
📊 预测结果 (Top 3): Surprised: 80.15%, Fearful: 8.27%, Angry: 5.47%

🎤 测试音频: en-US-GuyNeural_cheerful_2x.wav
📊 预测结果 (Top 3): Surprised: 79.55%, Fearful: 8.38%, Angry: 5.72%

🎤 测试音频: en-US-NancyNeural_cheerful_2x.wav
📊 预测结果 (Top 3): Surprised: 79.88%, Fearful: 8.33%, Angry: 5.63%

📂 进入文件夹: sad_sentence9，找到 4 个音频文件
📝 对应文本: "You have got to be kidding me."

🎤 测试音频: en-US-AriaNeural_sad_2x.wav
📊 预测结果 (Top 3): Surprised: 79.39%, Fearful: 8.49%, Angry: 5.76%

🎤 测试音频: en-US-DavisNeural_sad_2x.wav
📊 预测结果 (Top 3): Surprised: 80.72%, Fearful: 8.10%, Angry: 5.32%

🎤 测试音频: en-US-GuyNeural_sad_2x.wav
📊 预测结果 (Top 3): Surprised: 79.59%, Fearful: 8.38%, Angry: 5.68%

🎤 测试音频: en-US-NancyNeural_sad_2x.wav
📊 预测结果 (Top 3): Surprised: 79.53%, Fearful: 8.42%, Ang